---
title: 06 Demonstrate Enriched YouTube Exposure Measures
---

This notebook shows what becomes possible once donated YouTube watch histories have been cleaned, linked to scraped platform metadata, and annotated with event-level indicators.

The goal is demonstration rather than new measurement. We use the enriched watch-event table from notebook 04, plus the watch-time estimates from notebook 05 when available, to show how one watch event can now carry donation fields, metadata fields, pathway indicators, format labels, day/night labels, estimated watch time, and problematic-view labels side by side. Then we create a few compact figures that mirror the kinds of claims discussed in the article manuscript.

The notebook reads:

- `outputs/tables/video_histories_enriched.csv`
- `outputs/tables/video_histories_watchtime.csv` when available
- `outputs/tables/exposure_linkage_report.csv` when available

It writes:

- `outputs/tables/demo_enriched_event_examples.csv`
- `outputs/tables/demo_measure_summary.csv`
- `outputs/figures/demo_pathway_mix.png`
- `outputs/figures/demo_participant_measure_shares.png`
- `outputs/figures/demo_daynight_shorts.png`
- `outputs/figures/demo_problematic_views.png`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})


## Locate Project Files

The notebook can be run from the project root or from inside the `scripts` folder. All printed paths are project-relative so rendered output does not expose private local folders.


In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        has_quarto_project = (candidate / "_quarto.yml").exists()
        has_enriched_table = (candidate / "outputs" / "tables" / "video_histories_enriched.csv").exists()
        if has_quarto_project and has_enriched_table:
            return candidate
    raise FileNotFoundError(
        "Could not find the Quarto project root. Run this notebook from "
        "inside youtube_donation_dsa_method after notebook 04 has created the enriched table."
    )


PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

ENRICHED_VIDEO_HISTORIES_PATH = OUTPUT_DIR / "video_histories_enriched.csv"
WATCHTIME_EVENTS_PATH = OUTPUT_DIR / "video_histories_watchtime.csv"
EXPOSURE_LINKAGE_REPORT_PATH = OUTPUT_DIR / "exposure_linkage_report.csv"
DEMO_EVENT_EXAMPLES_PATH = OUTPUT_DIR / "demo_enriched_event_examples.csv"
DEMO_MEASURE_SUMMARY_PATH = OUTPUT_DIR / "demo_measure_summary.csv"

FIGURE_PATHS = {
    "pathway_mix": FIGURE_DIR / "demo_pathway_mix.png",
    "participant_measure_shares": FIGURE_DIR / "demo_participant_measure_shares.png",
    "daynight_shorts": FIGURE_DIR / "demo_daynight_shorts.png",
    "problematic_views": FIGURE_DIR / "demo_problematic_views.png",
}

print(f"Project folder: {PROJECT_ROOT.name}")
print(f"Enriched input: {ENRICHED_VIDEO_HISTORIES_PATH.relative_to(PROJECT_ROOT).as_posix()}")
if WATCHTIME_EVENTS_PATH.exists():
    print(f"Watch-time input: {WATCHTIME_EVENTS_PATH.relative_to(PROJECT_ROOT).as_posix()}")
else:
    print("Watch-time input: not found; continuing without estimated watch-time columns")
print(f"Figure folder: {FIGURE_DIR.relative_to(PROJECT_ROOT).as_posix()}")


## Load The Enriched Event Table

The input table is event-level: each row is one watch event. The columns added by notebook 04 are treated as the source of truth here. This notebook reshapes them for display and plotting, but it does not create new classifications.


In [ ]:
def ensure_columns(dataframe, required_columns, label):
    missing_columns = sorted(set(required_columns) - set(dataframe.columns))
    if missing_columns:
        raise ValueError(
            f"Missing columns in {label}: {missing_columns}. "
            f"Found: {list(dataframe.columns)}"
        )


def to_nullable_bool(series):
    normalized = series.astype("string").str.strip().str.lower()
    mapped = normalized.map(
        {
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "yes": True,
            "no": False,
        }
    )
    return mapped.astype("boolean")


video_histories_enriched = pd.read_csv(ENRICHED_VIDEO_HISTORIES_PATH, low_memory=False)
video_histories_enriched["watch_event_position"] = range(len(video_histories_enriched))

required_columns = {
    "Participant ID",
    "time",
    "watch_local_time",
    "video_id",
    "clean_title",
    "url",
    "metadata_title",
    "metadata_channel",
    "duration_seconds",
    "view_count",
    "metadata_status",
    "searched_for",
    "subscribed",
    "is_short",
    "time_of_day",
    "generalized_error_msg",
    "problematic_category",
    "problematic_assessment_label",
    "is_problematic",
}
ensure_columns(video_histories_enriched, required_columns, "video_histories_enriched.csv")

row_count = len(video_histories_enriched)
video_histories_enriched["time"] = pd.to_datetime(
    video_histories_enriched["time"],
    utc=True,
    format="mixed",
    errors="coerce",
)
video_histories_enriched["watch_local_time"] = pd.to_datetime(
    video_histories_enriched["watch_local_time"],
    format="mixed",
    errors="coerce",
)

for boolean_column in ["searched_for", "subscribed", "is_short", "is_problematic"]:
    video_histories_enriched[boolean_column] = to_nullable_bool(video_histories_enriched[boolean_column])

if video_histories_enriched["time"].isna().any():
    raise ValueError("Some rows have invalid UTC watch times.")
if len(video_histories_enriched) != row_count:
    raise AssertionError("Loading changed the number of enriched watch events.")

watchtime_available = WATCHTIME_EVENTS_PATH.exists()
if watchtime_available:
    watchtime_events = pd.read_csv(
        WATCHTIME_EVENTS_PATH,
        usecols=[
            "watch_event_position",
            "estimated_watch_seconds",
            "estimated_watch_minutes",
            "estimated_watch_hours",
            "watchtime_status",
        ],
        low_memory=False,
    )
    video_histories_enriched = video_histories_enriched.merge(
        watchtime_events,
        on="watch_event_position",
        how="left",
        validate="one_to_one",
    )
else:
    watchtime_events = pd.DataFrame()

input_summary = pd.DataFrame(
    {
        "measure": [
            "watch_events",
            "participants",
            "unique_videos",
            "earliest_watch_utc",
            "latest_watch_utc",
            "watchtime_estimates_available",
        ],
        "value": [
            row_count,
            video_histories_enriched["Participant ID"].nunique(),
            video_histories_enriched["video_id"].nunique(),
            video_histories_enriched["time"].min().isoformat(),
            video_histories_enriched["time"].max().isoformat(),
            str(watchtime_available),
        ],
    }
)

input_summary


## Prepare Reader-Friendly Labels

The columns below are display helpers for the showcase and figures. They keep the underlying event flags unchanged, but make the notebook easier to read.


In [ ]:
def shorten_text(value, max_chars=72):
    if pd.isna(value):
        return pd.NA
    text = " ".join(str(value).split())
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 3].rstrip() + "..."


def status_label(value):
    if pd.isna(value):
        return "Unknown"
    if value is pd.NA:
        return "Unknown"
    return "Yes" if bool(value) else "No"


def subscribed_label(value):
    if pd.isna(value):
        return "Unknown"
    return "Yes" if bool(value) else "No"


def format_number(value):
    if pd.isna(value):
        return pd.NA
    return f"{int(value):,}"


analysis_events = video_histories_enriched.copy()
analysis_events["searched_confirmed"] = analysis_events["searched_for"].eq(True)
analysis_events["subscribed_confirmed"] = analysis_events["subscribed"].eq(True)
analysis_events["short_confirmed"] = analysis_events["is_short"].eq(True)
analysis_events["problematic_confirmed"] = analysis_events["is_problematic"].eq(True)
analysis_events["night_confirmed"] = analysis_events["time_of_day"].astype("string").str.lower().eq("night")
analysis_events["video_format"] = "Longs"
analysis_events.loc[analysis_events["short_confirmed"], "video_format"] = "Shorts"
analysis_events["time_period"] = analysis_events["time_of_day"].astype("string").str.title()
analysis_events["pathway"] = "Neither search nor subscription"
analysis_events.loc[
    analysis_events["searched_confirmed"] & ~analysis_events["subscribed_confirmed"],
    "pathway",
] = "Search-associated only"
analysis_events.loc[
    ~analysis_events["searched_confirmed"] & analysis_events["subscribed_confirmed"],
    "pathway",
] = "Subscribed-channel only"
analysis_events.loc[
    analysis_events["searched_confirmed"] & analysis_events["subscribed_confirmed"],
    "pathway",
] = "Both search and subscription"

analysis_events["clean_title_short"] = analysis_events["clean_title"].map(shorten_text)
analysis_events["metadata_title_short"] = analysis_events["metadata_title"].map(shorten_text)
analysis_events["generalized_error_msg_short"] = analysis_events["generalized_error_msg"].map(shorten_text)
analysis_events["duration_minutes"] = (pd.to_numeric(analysis_events["duration_seconds"], errors="coerce") / 60).round(1)
if "estimated_watch_minutes" in analysis_events.columns:
    analysis_events["estimated_watch_minutes"] = pd.to_numeric(
        analysis_events["estimated_watch_minutes"], errors="coerce"
    ).round(1)
analysis_events["view_count_display"] = pd.to_numeric(analysis_events["view_count"], errors="coerce").map(format_number)

analysis_events[["pathway", "video_format", "time_period"]].head()


## What The Enriched Data Now Makes Visible

The table below is a small set of illustrative watch events. It is meant to show the shape of the enriched data before we aggregate anything: a donated watch event can now be read together with scraped metadata, pathway context, format labels, day/night timing, and problematic-view signals.

The participant names are safe here because the data are mock data. In a real donation study, this section should be pseudonymized, aggregated, or suppressed if small cells could reveal sensitive viewing behavior.


In [ ]:
example_specs = [
    ("Search-associated watch", analysis_events["searched_confirmed"]),
    ("Subscribed-channel watch", analysis_events["subscribed_confirmed"]),
    ("Shorts watch", analysis_events["short_confirmed"]),
    ("Nighttime watch", analysis_events["night_confirmed"]),
    ("Problematic/unavailable signal", analysis_events["problematic_confirmed"]),
    (
        "Neither search nor subscription",
        analysis_events["pathway"].eq("Neither search nor subscription")
        & ~analysis_events["problematic_confirmed"],
    ),
]

selected_rows = []
used_indices = set()
for reason, mask in example_specs:
    candidates = analysis_events[mask].copy()
    if candidates.empty:
        continue
    unused_candidates = candidates.loc[~candidates.index.isin(used_indices)]
    chosen = unused_candidates.head(1) if not unused_candidates.empty else candidates.head(1)
    chosen = chosen.copy()
    chosen["showcase_reason"] = reason
    selected_rows.append(chosen)
    used_indices.add(chosen.index[0])

if not selected_rows:
    raise ValueError("No showcase rows could be selected from the enriched table.")

showcase_columns = [
    "showcase_reason",
    "Participant ID",
    "watch_local_time",
    "clean_title_short",
    "url",
    "metadata_title_short",
    "metadata_channel",
    "duration_minutes",
    "view_count_display",
    "searched_for",
    "subscribed",
    "video_format",
    "time_period",
    "metadata_status",
    "generalized_error_msg_short",
    "problematic_assessment_label",
]
if "estimated_watch_minutes" in analysis_events.columns:
    insert_after = showcase_columns.index("duration_minutes") + 1
    showcase_columns.insert(insert_after, "estimated_watch_minutes")

demo_enriched_event_examples = pd.concat(selected_rows, ignore_index=True)[showcase_columns].rename(
    columns={
        "Participant ID": "participant_id",
        "watch_local_time": "watch_local_time",
        "clean_title_short": "clean_title",
        "metadata_title_short": "metadata_title",
        "metadata_channel": "channel",
        "duration_minutes": "duration_minutes",
        "estimated_watch_minutes": "estimated_watch_minutes",
        "view_count_display": "view_count",
        "searched_for": "searched",
        "subscribed": "subscribed",
        "generalized_error_msg_short": "generalized_error_msg",
    }
)

demo_enriched_event_examples["searched"] = demo_enriched_event_examples["searched"].map(status_label)
demo_enriched_event_examples["subscribed"] = demo_enriched_event_examples["subscribed"].map(subscribed_label)
demo_enriched_event_examples["watch_local_time"] = demo_enriched_event_examples["watch_local_time"].astype("string")

demo_enriched_event_examples


## Build Compact Summary Tables

These tables turn the event-level columns into aggregate measures. The goal is to make the denominator visible: the shares below are shares of watch events in the enriched table.


In [ ]:
watch_events = len(analysis_events)

def event_count_share(label, count):
    return {
        "section": "whole_dataset",
        "measure": label,
        "value": int(count),
        "share_of_watch_events": count / watch_events if watch_events else pd.NA,
    }


summary_rows = [
        {"section": "whole_dataset", "measure": "watch_events", "value": watch_events, "share_of_watch_events": pd.NA},
        {"section": "whole_dataset", "measure": "participants", "value": analysis_events["Participant ID"].nunique(), "share_of_watch_events": pd.NA},
        {"section": "whole_dataset", "measure": "unique_videos", "value": analysis_events["video_id"].nunique(), "share_of_watch_events": pd.NA},
        event_count_share("searched_watch_events", analysis_events["searched_confirmed"].sum()),
        event_count_share("confirmed_subscribed_watch_events", analysis_events["subscribed_confirmed"].sum()),
        event_count_share("shorts_watch_events", analysis_events["short_confirmed"].sum()),
        event_count_share("nighttime_watch_events", analysis_events["night_confirmed"].sum()),
        event_count_share("problematic_watch_events", analysis_events["problematic_confirmed"].sum()),
    ]

if "estimated_watch_hours" in analysis_events.columns:
    summary_rows.append({
        "section": "whole_dataset",
        "measure": "total_estimated_watch_hours",
        "value": round(float(pd.to_numeric(analysis_events["estimated_watch_hours"], errors="coerce").sum(skipna=True)), 3),
        "share_of_watch_events": pd.NA,
    })

demo_measure_summary = pd.DataFrame(summary_rows)

participant_aggregations = dict(
    watch_events=("video_id", "size"),
    searched_events=("searched_confirmed", "sum"),
    subscribed_events=("subscribed_confirmed", "sum"),
    shorts_events=("short_confirmed", "sum"),
    nighttime_events=("night_confirmed", "sum"),
    problematic_events=("problematic_confirmed", "sum"),
)
if "estimated_watch_hours" in analysis_events.columns:
    analysis_events["estimated_watch_hours"] = pd.to_numeric(analysis_events["estimated_watch_hours"], errors="coerce")
    participant_aggregations["estimated_watch_hours"] = ("estimated_watch_hours", "sum")

participant_summary = (
    analysis_events.groupby("Participant ID", dropna=False)
    .agg(**participant_aggregations)
    .reset_index()
    .rename(columns={"Participant ID": "participant_id"})
)

for count_column, share_column in [
    ("searched_events", "searched_share"),
    ("subscribed_events", "subscribed_share"),
    ("shorts_events", "shorts_share"),
    ("nighttime_events", "nighttime_share"),
    ("problematic_events", "problematic_share"),
]:
    participant_summary[share_column] = participant_summary[count_column] / participant_summary["watch_events"]

pathway_order = [
    "Neither search nor subscription",
    "Search-associated only",
    "Subscribed-channel only",
    "Both search and subscription",
]
pathway_summary = (
    analysis_events["pathway"]
    .value_counts()
    .reindex(pathway_order, fill_value=0)
    .rename_axis("pathway")
    .reset_index(name="watch_events")
)
pathway_summary["share_of_watch_events"] = pathway_summary["watch_events"] / watch_events

format_order = ["Shorts", "Longs"]
time_order = ["Day", "Night"]
daynight_format_summary = pd.crosstab(
    analysis_events["time_period"],
    analysis_events["video_format"],
).reindex(index=time_order, columns=format_order, fill_value=0)

problematic_assessment_summary = (
    analysis_events["problematic_assessment_label"]
    .value_counts(dropna=False)
    .rename_axis("problematic_assessment_label")
    .reset_index(name="watch_events")
)
problematic_assessment_summary["share_of_watch_events"] = problematic_assessment_summary["watch_events"] / watch_events

problematic_category_summary = (
    analysis_events["problematic_category"]
    .value_counts(dropna=False)
    .rename_axis("problematic_category")
    .reset_index(name="watch_events")
)
problematic_category_summary["share_of_watch_events"] = problematic_category_summary["watch_events"] / watch_events

share_columns = [column for column in participant_summary.columns if column.endswith("_share")]
if not participant_summary[share_columns].map(lambda value: 0 <= value <= 1).all().all():
    raise AssertionError("Participant-level shares should be between 0 and 1.")

if len(analysis_events) != row_count:
    raise AssertionError("Summary construction changed the enriched input row count.")

demo_measure_summary


## Figure 1: Pathway Mix

This figure illustrates the manuscript's pathway question: how much watched content appears to be associated with search, with subscribed channels, with both, or with neither observed route? The categories are indicators, not causal explanations of how YouTube produced the view.


In [ ]:
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plot_data = pathway_summary.sort_values("watch_events", ascending=True)
fig, ax = plt.subplots(figsize=(8, 4.6))
colors = ["#6b7280", "#2f6f9f", "#c17c2c", "#4f8f5b"]
ax.barh(plot_data["pathway"], plot_data["watch_events"], color=colors[: len(plot_data)])
ax.set_title("Watch Events By Observed Pathway")
ax.set_xlabel("Watch events")
ax.set_ylabel("")
for index, row in plot_data.reset_index(drop=True).iterrows():
    ax.text(
        row["watch_events"] + 2,
        index,
        f"{row['watch_events']} ({row['share_of_watch_events']:.0%})",
        va="center",
        fontsize=9,
    )
ax.set_xlim(0, max(plot_data["watch_events"]) * 1.2)
fig.tight_layout()
fig.savefig(FIGURE_PATHS["pathway_mix"], bbox_inches="tight")
plt.show()

pathway_summary


## Figure 2: Participant-Level Measure Shares

The same event-level flags can also be summarized by participant. In the mock data this is safe to display; with real donations, participant-level outputs should be pseudonymized and checked for disclosure risk.


In [ ]:
measure_specs = [
    ("searched_share", "Searched"),
    ("subscribed_share", "Subscribed"),
    ("shorts_share", "Shorts"),
    ("nighttime_share", "Night"),
    ("problematic_share", "Problematic"),
]
participants = participant_summary["participant_id"].tolist()
x_positions = list(range(len(participants)))
bar_width = 0.15
palette = ["#2f6f9f", "#c17c2c", "#4f8f5b", "#7f6a93", "#b4473a"]

fig, ax = plt.subplots(figsize=(9, 4.8))
for measure_index, (column, label) in enumerate(measure_specs):
    offset = (measure_index - (len(measure_specs) - 1) / 2) * bar_width
    positions = [x + offset for x in x_positions]
    ax.bar(
        positions,
        participant_summary[column],
        width=bar_width,
        label=label,
        color=palette[measure_index],
    )

ax.set_title("Selected Exposure Measures By Mock Participant")
ax.set_ylabel("Share of watch events")
ax.set_xticks(x_positions)
ax.set_xticklabels(participants)
ax.set_ylim(0, max(0.3, participant_summary[[column for column, _ in measure_specs]].max().max() * 1.25))
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
ax.legend(ncol=3, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12))
fig.tight_layout()
fig.savefig(FIGURE_PATHS["participant_measure_shares"], bbox_inches="tight")
plt.show()

participant_summary


## Figure 3: Day/Night And Shorts/Longs

This figure combines the locally interpreted watch time with metadata-derived video format. It shows how routine-oriented questions can be crossed with content-format indicators once timestamps and metadata sit in the same event table.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.6))
bottom_values = [0] * len(daynight_format_summary)
format_colors = {"Shorts": "#4f8f5b", "Longs": "#6b7280"}

for format_label in format_order:
    values = daynight_format_summary[format_label].tolist()
    ax.bar(
        daynight_format_summary.index,
        values,
        bottom=bottom_values,
        label=format_label,
        color=format_colors[format_label],
    )
    bottom_values = [bottom + value for bottom, value in zip(bottom_values, values)]

ax.set_title("Day And Night Watching By Video Format")
ax.set_ylabel("Watch events")
ax.set_xlabel("")
ax.legend(frameon=False)
for index, total in enumerate(bottom_values):
    ax.text(index, total + 3, str(total), ha="center", fontsize=9)
ax.set_ylim(0, max(bottom_values) * 1.18)
fig.tight_layout()
fig.savefig(FIGURE_PATHS["daynight_shorts"], bbox_inches="tight")
plt.show()

daynight_format_summary


## Figure 4: Problematic And Unavailable Signals

The problematic-view labels come from metadata errors and the manual classification table used in notebook 04. They should be read as later availability and classification signals, not as a direct content analysis of what the participant saw at the time of watching.


In [ ]:
plot_data = problematic_assessment_summary.sort_values("watch_events", ascending=True)
fig, ax = plt.subplots(figsize=(8, 4.6))
assessment_colors = ["#6b7280", "#2f6f9f", "#b4473a", "#c17c2c"]
ax.barh(
    plot_data["problematic_assessment_label"],
    plot_data["watch_events"],
    color=assessment_colors[: len(plot_data)],
)
ax.set_title("Watch Events By Availability / Problematic Assessment")
ax.set_xlabel("Watch events")
ax.set_ylabel("")
for index, row in plot_data.reset_index(drop=True).iterrows():
    ax.text(
        row["watch_events"] + 2,
        index,
        f"{row['watch_events']} ({row['share_of_watch_events']:.0%})",
        va="center",
        fontsize=9,
    )
ax.set_xlim(0, max(plot_data["watch_events"]) * 1.2)
fig.tight_layout()
fig.savefig(FIGURE_PATHS["problematic_views"], bbox_inches="tight")
plt.show()

problematic_category_summary


## Save Demo Outputs

The CSV outputs keep the compact showcase table and whole-dataset summary. The figures are saved as standalone PNG files so they can be used in the manuscript, slides, or a rendered Quarto document.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

demo_enriched_event_examples.to_csv(DEMO_EVENT_EXAMPLES_PATH, index=False)
demo_measure_summary.to_csv(DEMO_MEASURE_SUMMARY_PATH, index=False)

for path in [DEMO_EVENT_EXAMPLES_PATH, DEMO_MEASURE_SUMMARY_PATH, *FIGURE_PATHS.values()]:
    if not path.exists():
        raise AssertionError(f"Expected output was not created: {path.name}")

saved_outputs = pd.DataFrame(
    {
        "output": [
            DEMO_EVENT_EXAMPLES_PATH.relative_to(PROJECT_ROOT).as_posix(),
            DEMO_MEASURE_SUMMARY_PATH.relative_to(PROJECT_ROOT).as_posix(),
            *[path.relative_to(PROJECT_ROOT).as_posix() for path in FIGURE_PATHS.values()],
        ]
    }
)

saved_outputs


## Result

The notebook stores a compact showcase table, a whole-dataset summary table, and four figures. The previews below are the main handoff for the article-facing demonstration.


In [ ]:
print("demo_enriched_event_examples.csv")
display(demo_enriched_event_examples.head(10))

print("demo_measure_summary.csv")
display(demo_measure_summary.head(10))

print("Participant summary")
display(participant_summary)

print("Pathway summary")
display(pathway_summary)

print("Problematic assessment summary")
display(problematic_assessment_summary)

print("Saved outputs")
for path in [DEMO_EVENT_EXAMPLES_PATH, DEMO_MEASURE_SUMMARY_PATH, *FIGURE_PATHS.values()]:
    print(path.relative_to(PROJECT_ROOT).as_posix())
